# 01 — Ingest and QC

Load Sackmann-style CSVs from `TML_DATA_DIR`, record manifest lineage, and inspect completion flags.
All logic lives in `tml.data` — this notebook only orchestrates calls.

In [11]:
from pathlib import Path

import pandas as pd

from tml.data.identity import PlayerIdentityMap
from tml.data.ingest import ingest_match_files
from tml.data.modeling_table import build_modeling_table
from tml.shared.config import get_settings

In [12]:
settings = get_settings()
root = settings.tml_data_dir
print(f"Data root: {root.resolve()}")

Data root: /Users/alexgonzalez/Documents/tennis moneyline predictions/notebooks/tml-data


In [16]:
# Adjust globs for the years you want to load.
# atp_paths = sorted(root.glob("2024.csv"))
# challenger_paths = sorted(root.glob("2024_challenger.csv"))

atp_paths = sorted(
    p for p in root.glob("*.csv")
    if p.name[:4].isdigit()
    and 2000 <= int(p.name[:4])
    and "_challenger" not in p.name
    and "_wta" not in p.name
)
challenger_paths = sorted(
    p for p in root.glob("*_challenger.csv")
    if p.name[:4].isdigit() and 2000 <= int(p.name[:4])
)

atp = ingest_match_files(atp_paths, root=root, tour_level="atp")
challenger = ingest_match_files(challenger_paths, root=root, tour_level="challenger") if challenger_paths else None

print("ATP snapshot:", atp.dataset_snapshot_id)
print("Manifest rows:", len(atp.manifest))

ATP snapshot: 4973d601dc08ea2399b4fe2364bf1bb977189d235b23aaa27ef4a9541f5f8d75
Manifest rows: 27


In [17]:
matches = atp.matches if challenger is None else pd.concat([atp.matches, challenger.matches], ignore_index=True)
matches["completion_status"].value_counts()

completion_status
completed     193705
retirement      6670
walkover         915
default           16
unknown            2
Name: count, dtype: int64

In [18]:
identity = PlayerIdentityMap()
modeling = build_modeling_table(matches, identity)
modeling.head()

,match_id,tourney_id,tourney_date,surface,best_of,tour_level,player_a_id,player_b_id,y_complete_win,prediction_regime,...,age_a,age_b,a_svpt,b_svpt,a_1stIn,b_1stIn,a_1stWon,b_1stWon,a_2ndWon,b_2ndWon
0,atp:2000-7308:1,2000-7308,2000-01-03,Hard,3,atp,C487,E113,0,pre_tournament,...,22.045,25.810,59.0,66.0,37.0,29.0,25.0,23.0,13.0,23.0
1,atp:2000-7308:2,2000-7308,2000-01-03,Hard,3,atp,F324,K260,1,pre_tournament,...,18.404,24.882,46.0,42.0,28.0,15.0,24.0,13.0,12.0,12.0
2,atp:2000-7308:3,2000-7308,2000-01-03,Hard,3,atp,A202,G352,0,pre_tournament,...,28.797,22.585,103.0,81.0,59.0,40.0,49.0,35.0,22.0,28.0
3,atp:2000-7308:4,2000-7308,2000-01-03,Hard,3,atp,G379,I052,1,pre_tournament,...,21.599,23.710,66.0,49.0,35.0,22.0,28.0,12.0,14.0,8.0
4,atp:2000-7308:5,2000-7308,2000-01-03,Hard,3,atp,D270,N250,0,pre_tournament,...,25.580,23.595,73.0,52.0,40.0,32.0,25.0,26.0,16.0,12.0


In [19]:
from pathlib import Path

out = Path("../data/processed/modeling.parquet")
out.parent.mkdir(parents=True, exist_ok=True)
modeling.to_parquet(out)
print("Wrote", out.resolve(), "rows=", len(modeling))

Wrote /Users/alexgonzalez/Documents/tennis moneyline predictions/data/processed/modeling.parquet rows= 193705
